In [3]:
"""
detect_and_link.py

Step 3 of the glass pipeline: detect particles in every frame
(trackpy.locate / trackpy.batch) and link detections across frames
into continuous trajectories (trackpy.link).

This script is built to BENCHMARK before committing to a full run:
given how much bigger this dataset is than the two inherited notebooks
(1500x1500 px, 4000 frames -- roughly 90x more raw detection work than
the largest of the two old notebooks), running blind on the full video
could take anywhere from tens of minutes to hours. So:

  TEST_MODE = True   -> runs on a small slice (N_TEST_FRAMES), times it,
                         and prints a projected estimate for the full run.
  TEST_MODE = False  -> runs the full pipeline on FRAME_START:FRAME_END.

Recommended workflow:
  1. Leave TEST_MODE = True. Tune DIAMETER / MINMASS / INVERT by eye
     using the preview image this produces, and check the projected
     full-run time.
  2. Once params look right and the time estimate is acceptable,
     set TEST_MODE = False and run the full thing (consider doing this
     overnight, or increasing N_PROCESSES to use more CPU cores).

Input:  the denoised .avi from denoise_background_subtract.py
Output: a .csv of linked particle trajectories (frame, x, y, particle ID)
"""

import time
import os
import cv2
import numpy as np
import pandas as pd
import trackpy as tp

# ----------------------------------------------------------------------
# 1. CONFIG -- edit these values
# ----------------------------------------------------------------------

T_PATH = "/Volumes/Expansion/recordings/denoised.avi"  
OUTPUT_DIR = os.path.dirname(INPUT_PATH)
OUTPUT_FILENAME = "linked_trajectories.csv"

# ---- trackpy.locate parameters ----
# DIAMETER must be an odd integer, roughly the particle's diameter in px.
# Tune these using the TEST_MODE preview image before trusting a full run.
DIAMETER = 11        # <-- check against your denoised preview frame
MINMASS = 200         # <-- minimum integrated brightness to count as a particle
SEPARATION = None     # None -> trackpy defaults to something reasonable given DIAMETER
INVERT = False        # True if particles are DARKER than background, False if BRIGHTER

# ---- trackpy.link parameters ----
# MAX_DISPLACEMENT was originally 5px, but diagnose_trajectory_breaks.py
# showed ~91,000 trajectories breaking, with a median gap of just 1 frame
# and median distance ~5.35px -- i.e. particles routinely move a hair more
# than 5px per frame, causing trackpy to refuse the link almost constantly.
# Raised to 9px to cover most of that. NOTE: this hasn't been checked
# against typical nearest-neighbor spacing yet (that comes from the
# density-stats step) -- if 9px turns out to be too close to typical
# inter-particle distance in crowded regions, this risks silently linking
# to the WRONG neighboring particle instead of leaving a broken track.
# Worth revisiting once nearest-neighbor stats exist.
MAX_DISPLACEMENT = 9  # max px a particle can move between consecutive frames
MEMORY = 3            # frames a particle can "disappear" for and still be linked

# ---- run mode ----
TEST_MODE = False
N_TEST_FRAMES = 50

FRAME_START = 0
FRAME_END = 4000

# ---- parallelism (only used when TEST_MODE = False) ----
N_PROCESSES = 1  # set higher (e.g. 4) to use multiple CPU cores via tp.batch


def load_frames(path, start, end):
    """Stream frames [start, end) from a video into a list of 2D arrays."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = min(end, total_frames)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for i in range(start, end):
        ret, frame = cap.read()
        if not ret:
            print(f"WARNING: failed to read frame {i}; stopping at {i}.")
            break
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(frame)
    cap.release()
    return frames


def run_detection_and_linking(frames, frame_offset, print_every=25):
    """
    Detect particles in each frame, then link into trajectories.
    frame_offset lets us record true frame numbers even when `frames`
    is a slice that doesn't start at 0.

    Prints a progress update (frame index, elapsed time, particles
    found so far) every `print_every` frames, plus a final summary.
    """
    all_features = []
    n_frames = len(frames)
    t_start = time.time()

    for i, frame in enumerate(frames):
        f = tp.locate(
            frame,
            diameter=DIAMETER,
            minmass=MINMASS,
            separation=SEPARATION,
            invert=INVERT,
        )
        f["frame"] = i + frame_offset
        all_features.append(f)

        if (i + 1) % print_every == 0 or (i + 1) == n_frames:
            elapsed = time.time() - t_start
            rate = (i + 1) / elapsed if elapsed > 0 else float("nan")
            n_found_so_far = sum(len(x) for x in all_features)
            remaining = n_frames - (i + 1)
            eta = remaining / rate if rate > 0 else float("nan")
            print(f"  detection: frame {i+1}/{n_frames}  "
                  f"({elapsed:.1f}s elapsed, {rate:.2f} frames/s, "
                  f"ETA {eta:.1f}s)  "
                  f"{n_found_so_far} detections so far")

    features = pd.concat(all_features, ignore_index=True)

    print(f"Detection done: {len(features)} total detections across "
          f"{n_frames} frames. Starting linking...")
    t_link = time.time()

    linked = tp.link(
        features,
        search_range=MAX_DISPLACEMENT,
        memory=MEMORY,
    )

    print(f"Linking done in {time.time() - t_link:.1f}s. "
          f"{linked['particle'].nunique()} distinct trajectories found.")

    return features, linked


def main():
    if INPUT_PATH is None or OUTPUT_DIR is None:
        raise ValueError("Set INPUT_PATH and OUTPUT_DIR before running.")

    if TEST_MODE:
        print(f"TEST MODE: running on {N_TEST_FRAMES} frames "
              f"(frames {FRAME_START} to {FRAME_START + N_TEST_FRAMES})")
        frames = load_frames(INPUT_PATH, FRAME_START, FRAME_START + N_TEST_FRAMES)
        print(f"Loaded {len(frames)} test frames.")

        # ---- preview: draw detected circles on the first test frame ----
        f0 = tp.locate(
            frames[0], diameter=DIAMETER, minmass=MINMASS,
            separation=SEPARATION, invert=INVERT,
        )
        preview = cv2.cvtColor(frames[0], cv2.COLOR_GRAY2BGR)
        for _, row in f0.iterrows():
            cv2.circle(preview, (int(row["x"]), int(row["y"])), int(DIAMETER / 2),
                       (0, 255, 0), 1)
        preview_dir = os.path.join(OUTPUT_DIR, "detection_previews")
        os.makedirs(preview_dir, exist_ok=True)
        preview_path = os.path.join(preview_dir, "test_frame0_detections.png")
        cv2.imwrite(preview_path, preview)
        print(f"Detected {len(f0)} particles in frame 0. "
              f"Preview saved to:\n  {preview_path}")
        print("--> Check this image before trusting these parameters.")

        # ---- benchmark full detect+link on the small slice ----
        t0 = time.time()
        features, linked = run_detection_and_linking(frames, FRAME_START)
        elapsed = time.time() - t0

        per_frame = elapsed / len(frames)
        full_n_frames = FRAME_END - FRAME_START
        projected = per_frame * full_n_frames

        print(f"\nBenchmark: {elapsed:.1f}s for {len(frames)} frames "
              f"({per_frame:.3f}s/frame)")
        print(f"Projected time for full run ({full_n_frames} frames): "
              f"{projected/60:.1f} minutes ({projected/3600:.2f} hours)")
        print("\nAdjust DIAMETER/MINMASS/INVERT if the preview looks wrong, "
              "or set TEST_MODE = False to run the full pipeline.")
        return

    # ------------------------------------------------------------------
    # FULL RUN
    # ------------------------------------------------------------------
    print(f"FULL RUN: frames {FRAME_START} to {FRAME_END}")
    frames = load_frames(INPUT_PATH, FRAME_START, FRAME_END)
    print(f"Loaded {len(frames)} frames.")

    t0 = time.time()

    if N_PROCESSES > 1:
        # tp.batch parallelizes detection (not linking) across frames
        features = tp.batch(
            frames, diameter=DIAMETER, minmass=MINMASS,
            separation=SEPARATION, invert=INVERT,
            processes=N_PROCESSES,
        )
        features["frame"] = features["frame"] + FRAME_START
        linked = tp.link(features, search_range=MAX_DISPLACEMENT, memory=MEMORY)
    else:
        features, linked = run_detection_and_linking(frames, FRAME_START)

    elapsed = time.time() - t0
    print(f"Detection + linking took {elapsed/60:.1f} minutes.")

    n_trajectories = linked["particle"].nunique()
    print(f"Found {n_trajectories} distinct trajectories across "
          f"{linked['frame'].nunique()} frames.")

    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    linked.to_csv(output_path, index=False)
    print(f"Saved linked trajectories to:\n  {output_path}")

    #Seth Waz Here


if __name__ == "__main__":
    main()

Frame 3999: 3917 trajectories present.
Linking done in 92.6s. 105380 distinct trajectories found.
Detection + linking took 17.7 minutes.
Found 105380 distinct trajectories across 4000 frames.
Saved linked trajectories to:
  /Volumes/Expansion/recordings/linked_trajectories.csv
